In [ ]:
!pip install CoolProp
!pip install uncertainties

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 2.0 MB/s eta 0:00:00


In [ ]:
import numpy as np
from uncertainties import ufloat
from uncertainties.umath import *
import CoolProp.CoolProp as CP



In [ ]:
v = 2

In [ ]:
 if v == 1:
  dados_medidos = {
      "V_ab": ufloat(30.0, 30.0 * 0.03),      # Tensão [V] com incerteza de ±3%
      "I": ufloat(0.369, 0.369 * 0.03),       # Corrente [A] com incerteza de ±3%
      "L": ufloat(194.0, 0.5),                # Comprimento [mm] com incerteza de ±0.5 mm
      "d": ufloat(25.5, 0.1),                 # Diâmetro [mm] com incerteza de ±0.1 mm
      "T_s1_leituras": [ufloat(80.5, 1.0), ufloat(80.6, 1.0), ufloat(80.6, 1.0)],
      "T_s2_leituras": [ufloat(77.9, 1.0), ufloat(78.0, 1.0), ufloat(77.7, 1.0)],
      "T_s3_leituras": [ufloat(77.0, 1.0), ufloat(76.9, 1.0), ufloat(77.2, 1.0)],
      "T_amb_leituras": [ufloat(24.7, 1.0), ufloat(25.2, 1.0)],
  }


if v == 2:
  dados_medidos = {
      "V_ab": ufloat(30.0, 30.0 * 0.03),      # Tensão [V] com incerteza de ±3%
      "I": ufloat(0.37, 0.37 * 0.03),       # Corrente [A] com incerteza de ±3%
      "L": ufloat(194.0, 0.5),                # Comprimento [mm] com incerteza de ±0.5 mm
      "d": ufloat(25.3, 0.1),                 # Diâmetro [mm] com incerteza de ±0.1 mm
      "T_s1_leituras": [ufloat(79.9, 1.0)],
      "T_s2_leituras": [ufloat(77.2, 1.0)],
      "T_s3_leituras": [ufloat(76.4, 1.0)],
      "T_amb_leituras": [ufloat(24.4, 1.0)],
  }

constantes = {
    "g": ufloat(9.78,0.01),
    "epsilon": 0.56,
    "sigma": 5.67e-8,
    "P_atm": 692.0*133.322 # Pressão atmosférica em Pascal
}

In [ ]:
def processar_valores(dados):
    L_m = dados["L"] / 1000.0
    d_m = dados["d"] / 1000.0
    T_s1_avg_C = np.mean(dados["T_s1_leituras"])
    T_s2_avg_C = np.mean(dados["T_s2_leituras"])
    T_s3_avg_C = np.mean(dados["T_s3_leituras"])
    T_s_avg_C = np.mean([T_s1_avg_C, T_s2_avg_C, T_s3_avg_C])
    T_amb_avg_C = np.mean(dados["T_amb_leituras"])
    T_s_avg_K = T_s_avg_C + 273.15
    T_amb_avg_K = T_amb_avg_C + 273.15
    valores_processados = {
        "L_m": L_m, "d_m": d_m, "T_s_avg_C": T_s_avg_C,
        "T_amb_avg_C": T_amb_avg_C, "T_s_avg_K": T_s_avg_K, "T_amb_avg_K": T_amb_avg_K
    }
    return valores_processados

In [ ]:
def calcular_h_experimental(dados, params):
    Q_ponto = dados["V_ab"] * dados["I"]
    S = np.pi * params["d_m"] * params["L_m"]
    delta_T = params["T_s_avg_C"] - params["T_amb_avg_C"]
    h_exp = Q_ponto / (S * delta_T)
    return h_exp, Q_ponto, S, delta_T

In [ ]:
def obter_propriedades_ar(T_filme_K, P_pascal):
    T_nom = T_filme_K.nominal_value
    mu = CP.PropsSI('V', 'T', T_nom, 'P', P_pascal, 'Air')      # Viscosidade dinâmica [Pa.s]
    rho = CP.PropsSI('D', 'T', T_nom, 'P', P_pascal, 'Air')     # Densidade [kg/m^3]
    Pr = CP.PropsSI('Prandtl', 'T', T_nom, 'P', P_pascal, 'Air') # Número de Prandtl

    # Calcular viscosidade cinemática v = μ/ρ [m^2/s]
    nu = mu / rho


    return nu, Pr

In [ ]:
def calcular_h_teorico(params, constantes):
    T_f_K = (params["T_s_avg_K"] + params["T_amb_avg_K"]) / 2

    v, Pr = obter_propriedades_ar(T_f_K, constantes["P_atm"])

    beta = 1 / T_f_K
    Gr = (constantes["g"] * beta * (params["T_s_avg_K"] - params["T_amb_avg_K"]) * params["d_m"]**3) / (v**2)
    Ra = Gr * Pr

    h_conv = 0
    delta_T_C = params["T_s_avg_C"] - params["T_amb_avg_C"]

    # Verifica o regime de escoamento e aplica a fórmula correta da Tabela 7-2
    if 1e4 < Ra.nominal_value < 1e9:
        h_conv = 1.32 * (delta_T_C / params["d_m"])**0.25
    elif Ra.nominal_value >= 1e9:
        h_conv = 1.24 * (delta_T_C)**(1/3)
    else:
              h_conv = ufloat(0, 0)

    Ts_K = params["T_s_avg_K"]
    Tamb_K = params["T_amb_avg_K"]
    h_rad = constantes["epsilon"] * constantes["sigma"] * (Ts_K**2 + Tamb_K**2) * (Ts_K + Tamb_K)
    h_teorico = h_conv + h_rad

    return h_teorico, T_f_K, v, Pr, Gr, Ra, h_conv, h_rad

In [ ]:
if __name__ == "__main__":
    # Etapa 1: Processar os dados de entrada
    parametros_calculo = processar_valores(dados_medidos)

    # Etapa 2: Calcular o coeficiente experimental
    h_exp, Q_ponto, S, delta_T = calcular_h_experimental(dados_medidos, parametros_calculo)

    # Etapa 3: Calcular o coeficiente teórico
    h_teorico, T_f_K, v, Pr, Gr, Ra, h_conv, h_rad = calcular_h_teorico(parametros_calculo, constantes)

    # --- APRESENTAÇÃO DOS RESULTADOS ---
    print("="*65)

    print("--- 1. Parâmetros Iniciais Processados ---")
    print(f"Temperatura Média da Superfície: {parametros_calculo['T_s_avg_C']:.2f} °C")
    print(f"Temperatura Média Ambiente:      {parametros_calculo['T_amb_avg_C']:.2f} °C\n")

    print("--- 2. Análise Experimental ---")
    print(f"Taxa de Calor (Potência Elétrica), Q = {Q_ponto:.2f} W")
    print(f"Área da Superfície Lateral, S =       {S:.4uF} m^2")
    print(f"Diferença de Temperatura, ΔT =        {delta_T:.2f} °C")
    print(f"Coeficiente Experimental (h_exp) =    {h_exp:.2f} W/(m^2·°C)\n")

    print("--- 3. Análise Teórica ---")
    print(f"Temperatura de Filme, Tf =            {T_f_K:.2f} K")
    print(f"Viscosidade Cinemática (v) do Ar =    {v:.2e} m^2/s")
    print(f"Número de Prandtl (Pr) do Ar =        {Pr:.3f}")
    print(f"Número de Rayleigh, Ra =              {Ra:.2e}")
    print("-" * 25)
    print(f"Coeficiente de Convecção (h_conv) =   {h_conv:.2f} W/(m^2·°C)")
    print(f"Coeficiente de Radiação (h_rad) =     {h_rad:.2f} W/(m^2·°C)")
    print(f"Coeficiente Teórico (h_teórico) =     {h_teorico:.2f} W/(m^2·°C)\n")

    print("="*65)
    print(f"Coeficiente Experimental (h_exp):   {h_exp:.2f} W/(m^2·°C)")
    print(f"Coeficiente Teórico (h_teórico):    {h_teorico:.2f} W/(m^2·°C)")
    print("-" * 65)

--- 1. Parâmetros Iniciais Processados ---
Temperatura Média da Superfície: 77.83+/-0.58 °C
Temperatura Média Ambiente:      24.40+/-1.00 °C

--- 2. Análise Experimental ---
Taxa de Calor (Potência Elétrica), Q = 11.10+/-0.47 W
Área da Superfície Lateral, S =       0.01541957+/-0.00007276 m^2
Diferença de Temperatura, ΔT =        53.43+/-1.15 °C
Coeficiente Experimental (h_exp) =    13.47+/-0.64 W/(m^2·°C)

--- 3. Análise Teórica ---
Temperatura de Filme, Tf =            324.27+/-0.58 K
Viscosidade Cinemática (v) do Ar =    1.99e-05 m^2/s
Número de Prandtl (Pr) do Ar =        0.704
Número de Rayleigh, Ra =              (4.66+/-0.12)e+04
-------------------------
Coeficiente de Convecção (h_conv) =   8.95+/-0.05 W/(m^2·°C)
Coeficiente de Radiação (h_rad) =     4.36+/-0.02 W/(m^2·°C)
Coeficiente Teórico (h_teórico) =     13.31+/-0.04 W/(m^2·°C)

Coeficiente Experimental (h_exp):   13.47+/-0.64 W/(m^2·°C)
Coeficiente Teórico (h_teórico):    13.31+/-0.04 W/(m^2·°C)
------------------------